This is done to only 500 rows, we need to apply the same change to the original dataset. (This is only for testing/exploring purposes/)

#Imports + Read Data

In [1]:
!pip -q install pyreadstat

import pyreadstat
import pandas as pd
import numpy as np

In [2]:
df, meta = pyreadstat.read_sav("hsls_17_student_pets_sr_v1_0.sav")

**Convert .sav to .csv for ease** (If needed later)

In [3]:
df.to_csv("hsls_17_student_pets_sr_v1_0.csv")

#Cleaning columns

## remove -5's (Rows that are Null's)

In [4]:
df.head()

,STU_ID,SCH_ID,X1NCESID,X2NCESID,STRAT_ID,PSU,X2UNIV1,X2UNIV2A,X2UNIV2B,X3UNIV1,...,W5W1W2W3W4PSRECORDS191,W5W1W2W3W4PSRECORDS192,W5W1W2W3W4PSRECORDS193,W5W1W2W3W4PSRECORDS194,W5W1W2W3W4PSRECORDS195,W5W1W2W3W4PSRECORDS196,W5W1W2W3W4PSRECORDS197,W5W1W2W3W4PSRECORDS198,W5W1W2W3W4PSRECORDS199,W5W1W2W3W4PSRECORDS200
0,10001,-5,-5,-5,-5.0,-5.0,11,1.0,1.0,1111,...,0.0,2098.087446,1824.641398,0.0,2431.665487,0.0,0.0,2457.423209,0.0,2053.40787
1,10002,-5,-5,-5,-5.0,-5.0,11,1.0,1.0,1111,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.00000
2,10003,-5,-5,-5,-5.0,-5.0,11,1.0,1.0,1111,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.00000
3,10004,-5,-5,-5,-5.0,-5.0,10,1.0,7.0,1001,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.00000
4,10005,-5,-5,-5,-5.0,-5.0,11,1.0,1.0,1111,...,0.0,0.000000,0.000000,0.0,0.000000,0.0,0.0,0.000000,0.0,0.00000


In [5]:
# num of columns
len(df.columns)

9614

In [6]:
mask_all_minus5 = df.apply(
    lambda s: pd.to_numeric(s, errors="coerce").eq(-5).all()
)

df = df.loc[:, ~mask_all_minus5]

In [7]:
# num of columns
len(df.columns)

8682

**932 columns have the values of -5**

## remove columns starting with "W" (These columns represent weights which we can't benefit for our approach)

In [8]:
df = df.loc[:, ~df.columns.astype(str).str.startswith("W")]

In [9]:
len(df.columns)

3054

In [10]:
df.head()

,STU_ID,X2UNIV1,X2UNIV2A,X2UNIV2B,X3UNIV1,X4UNIV1,X1SEX,X1RACE,X1HISPANIC,X1WHITE,...,X5PFYNETPRICEGRT_IM,X5PFYPELLPACK_IM,X5PFYTOTLOAN_IM,X5PFYTOTLOAN2_IM,X5PFYTOTLOAN3_IM,X5EVRFEDAPP_IM,X5FEDAPP14_IM,X5FEDAPP15_IM,X5FEDAPP16_IM,X5PFYTUITION_IM
0,10001,11,1.0,1.0,1111,11111,1.0,8.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,10002,11,1.0,1.0,1111,11111,2.0,8.0,0.0,1.0,...,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0
2,10003,11,1.0,1.0,1111,11111,2.0,3.0,0.0,0.0,...,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0,-8.0
3,10004,10,1.0,7.0,1001,10011,2.0,8.0,0.0,1.0,...,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0
4,10005,11,1.0,1.0,1111,11111,1.0,8.0,0.0,1.0,...,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0,-6.0


**5628 Columns started with "W"**

##Delete columns in which end with _IM which are imputed values

In [11]:
df = df.loc[:, ~df.columns.str.endswith('_IM')]

In [12]:
len(df.columns)

2926

**119 Columns ended with "_IM"**

## Delete columns in which are from the 3rd wave.

In [23]:
df = df.loc[:, ~((df.columns.str.len() > 1) & (df.columns.str[1] == '3'))]

In [24]:
len(df.columns)

2637

## Delete columns in which are from the 4th wave (**Keeping our target**).

In [25]:
df = df.loc[:, ~((df.columns.str.len() > 1) & (df.columns.str[1] == '4') & (df.columns != 'X4EVERDROP'))]

In [26]:
len(df.columns)

2255

## Delete columns in which are from the 5th wave.

In [27]:
df = df.loc[:, ~((df.columns.str.len() > 1) & (df.columns.str[1] == '5'))]

In [28]:
len(df.columns)

2049

**We can choose columns from 2049 now instead of the whole original 9614 columns (way easier)**

In [30]:
# for reference
df.to_csv("hsls_17_students_no_minus_5_no_start_w_no_end_with_IM_only_1st_2nd_waves.csv")

# Cleaning some rows

**this step is needed fo the sake of the selection technique that we'll be using afterwards**

In [16]:
df.head()

,STU_ID,X2UNIV1,X2UNIV2A,X2UNIV2B,X3UNIV1,X4UNIV1,X1SEX,X1RACE,X1HISPANIC,X1WHITE,...,X4X2SES1,X4X2SES2,X4X2SES3,X4X2SES4,X4X2SES5,X4X2SES1_U,X4X2SES2_U,X4X2SES3_U,X4X2SES4_U,X4X2SES5_U
0,10001,11,1.0,1.0,1111,11111,1.0,8.0,0.0,1.0,...,1.5184,1.5184,1.5184,1.5184,1.5184,1.8774,1.8774,1.8774,1.8774,1.8774
1,10002,11,1.0,1.0,1111,11111,2.0,8.0,0.0,1.0,...,-0.1124,-0.1124,-0.1124,-0.1124,-0.1124,-0.0754,-0.0754,-0.0754,-0.0754,-0.0754
2,10003,11,1.0,1.0,1111,11111,2.0,3.0,0.0,0.0,...,0.9986,0.9986,0.9986,0.9986,0.9986,1.2472,1.2472,1.2472,1.2472,1.2472
3,10004,10,1.0,7.0,1001,10011,2.0,8.0,0.0,1.0,...,-0.2613,-0.2613,-0.2613,-0.2613,-0.2613,-0.2284,-0.2284,-0.2284,-0.2284,-0.2284
4,10005,11,1.0,1.0,1111,11111,1.0,8.0,0.0,1.0,...,1.1234,1.1234,1.1234,1.1234,1.1234,0.9926,0.9926,0.9926,0.9926,0.9926


In [17]:
# print(df.columns.tolist())

# Code to get the types of columns based on the first two letters
lst = df.columns.tolist()
s = set()

for name in lst:
  if name[:2] not in s:
    s.add(name[:2])

print(sorted(s))

['A1', 'A2', 'C1', 'C2', 'M1', 'N1', 'P1', 'P2', 'S1', 'S2', 'S3', 'S4', 'ST', 'X1', 'X2', 'X3', 'X4', 'X5']


**We need to know what each of these mean ['A1', 'A2', 'C1', 'C2', 'M1', 'N1', 'P1', 'P2', 'S1', 'S2', 'S3', 'S4', 'ST', 'X1', 'X2', 'X3', 'X4', 'X5']**

Prefixes:

**A1 / A2**: school administrator questionnaire items (school characteristics, scheduling, policies; principal background), replicated to student records for student-level analyses with school context.

**C1 / C2**: counselor questionnaire items (counseling programs/practices; placement; supports). This counselor data is contextual and linked to students in the school.

**M1 / N1**: base-year mathematics teacher and science teacher context items; teacher data intended as classroom context rather than student-rating data.

**P1 / P2**: base-year and first follow-up parent questionnaires (family context, SES components, involvement, expectations, resources).

**S1 / S2 / S3 / S4**: student instruments across waves (base year, first follow-up, 2013 update, second follow-up). HSLS second follow-up instrument mapping explicitly uses “S4 …” identifiers for its web survey structure.

**ST**: in many NCES codebooks, “ST” prefixes often denote “student” or “student transcript” segments; in HSLS documentation, transcript content is a major component, but the exact “ST…” variable namespace should be validated against the transcript codebooks/Online Codebook.

**X1–X5**: derived composites and cross-wave constructed variables (demographics composites, response-status encodings, enrollment-pathway constructs, imputation flags, and linkage identifiers). Clear examples include X1RACE, X2UNIV2B, and X3UNIV1.

Postfixes:
**_IM**: indicating whether an upstream value was imputed.

Converting all of the minus values into null values

-9: **Missing**

-8: **Unit non-response**

-7: **Item legitimate skip/NA**

-6: **Component not applicable**

-4: **Item not administered: abbreviated interview**

In [18]:
df = df.replace([-9, -8, -7, -6, -4], np.nan)

Seeing negative values that are valid

In [19]:
df_num = df.select_dtypes(include="number")
neg_value_counts = df_num.where(df_num < 0).stack().value_counts().sort_index()
neg_value_counts

,count
-6.4300,48
-6.0400,26
-5.6400,15
-5.5100,26
-5.4900,1
...,...
-0.0005,22
-0.0004,43
-0.0003,33
-0.0002,19


In [20]:
# Find columns with one unique value, excluding NaNs
one_value_cols_no_nan = df.nunique(dropna=True)[df.nunique(dropna=True) == 1].index.tolist()
print(f"Columns with only one unique value (excluding NaNs): {one_value_cols_no_nan}")

# Find columns with one unique value, including NaNs (note how 'D' and 'E' are included)
one_value_cols_incl_nan = df.nunique(dropna=False)[df.nunique(dropna=False) == 1].index.tolist()
print(f"Columns with only one unique value (including NaNs): {one_value_cols_incl_nan}")

Columns with only one unique value (excluding NaNs): []
Columns with only one unique value (including NaNs): []


**No columns which have only one value**

#Columns Choosing **Final** (Feature Selection)

In [21]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

In [29]:
df.head()

,STU_ID,X2UNIV1,X2UNIV2A,X2UNIV2B,X1SEX,X1RACE,X1HISPANIC,X1WHITE,X1BLACK,X1DUALLANG,...,X2SES1,X2SES2,X2SES3,X2SES4,X2SES5,X2SES1_U,X2SES2_U,X2SES3_U,X2SES4_U,X2SES5_U
0,10001,11,1.0,1.0,1.0,8.0,0.0,1.0,0.0,1.0,...,1.5649,1.5649,1.5649,1.5649,1.5649,1.6792,1.6792,1.6792,1.6792,1.6792
1,10002,11,1.0,1.0,2.0,8.0,0.0,1.0,0.0,1.0,...,-0.3307,-0.3307,-0.3307,-0.3307,-0.3307,-0.3234,-0.3234,-0.3234,-0.3234,-0.3234
2,10003,11,1.0,1.0,2.0,3.0,0.0,0.0,1.0,1.0,...,1.0141,1.0141,1.0141,1.0141,1.0141,0.8916,0.8916,0.8916,0.8916,0.8916
3,10004,10,1.0,7.0,2.0,8.0,0.0,1.0,0.0,1.0,...,-0.3347,-0.3347,-0.3347,-0.3347,-0.3347,-0.4367,-0.4367,-0.4367,-0.4367,-0.4367
4,10005,11,1.0,1.0,1.0,8.0,0.0,1.0,0.0,1.0,...,0.9661,0.9661,0.9661,0.9661,0.9661,1.0466,1.0466,1.0466,1.0466,1.0466


In [31]:
# this column causes high data leakge
df = df.drop('X2EVERDROP', axis=1)

In [32]:
print("=== X4EVERDROP value counts ===")
print(df['X4EVERDROP'].value_counts(dropna=False).sort_index())

=== X4EVERDROP value counts ===
X4EVERDROP
0.0    14618
1.0     2714
NaN     6171
Name: count, dtype: int64


Finding the most contributing columns to our target using:

- Mutual Information — measures non-linear statistical dependency with the target

- Random Forest Feature Importance — captures predictive power in a tree-based model

- Absolute Correlation — measures linear relationship strength with the target

In [33]:
TARGET = 'X4EVERDROP'

# Drop rows where target is missing
df_clean = df.dropna(subset=[TARGET])
print(f"Shape after dropping missing target: {df_clean.shape}")
print(f"Target value counts:\n{df_clean[TARGET].value_counts()}")

# Separate features and target
y = df_clean[TARGET].astype(int)
X = df_clean.drop(columns=[TARGET, 'STU_ID'], errors='ignore')

# Keep only numeric columns
X = X.select_dtypes(include=[np.number])

# Drop columns that are mostly NaN (>70% missing)
thresh = len(X) * 0.30
X = X.dropna(thresh=thresh, axis=1)
print(f"Features after dropping high-missing cols: {X.shape[1]}")

# Fill remaining NaN with median
X_filled = X.fillna(X.median())

# --- METHOD 1: Mutual Information ---
print("\nComputing Mutual Information...")
mi_scores = mutual_info_classif(X_filled, y, random_state=42)
mi_series = pd.Series(mi_scores, index=X_filled.columns).sort_values(ascending=False)

# --- METHOD 2: Random Forest Feature Importance ---
print("Training Random Forest...")
rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1, max_depth=8)
rf.fit(X_filled, y)
rf_series = pd.Series(rf.feature_importances_, index=X_filled.columns).sort_values(ascending=False)

# --- METHOD 3: Absolute Correlation with target ---
print("Computing Correlations...")
corr_series = X_filled.corrwith(y).abs().sort_values(ascending=False)

# --- Combine: Rank-based ensemble ---
# Rank each method (lower rank = more important)
mi_rank = mi_series.rank(ascending=False)
rf_rank = rf_series.rank(ascending=False)
corr_rank = corr_series.rank(ascending=False)

# Align all on same index
all_features = X_filled.columns
combined = pd.DataFrame({
    'MI_score': mi_series,
    'RF_importance': rf_series,
    'Corr_with_target': corr_series,
    'MI_rank': mi_rank,
    'RF_rank': rf_rank,
    'Corr_rank': corr_rank,
}, index=all_features)

combined['avg_rank'] = combined[['MI_rank', 'RF_rank', 'Corr_rank']].mean(axis=1)
combined = combined.sort_values('avg_rank')

top100 = combined.head(100)

print("\n=== TOP 100 FEATURES ===")
print(top100[['MI_score', 'RF_importance', 'Corr_with_target', 'avg_rank']].to_string())

top100.to_csv('top100_features_X4EVERDROP.csv')
print("\nSaved to outputs!")

# specific_cols = ['S2GRD1112', 'X1PAR1EDU', 'X1PAR2EDU', 'X1FAMINCOME', 'X1PAR1OCC2', 'X1PAR2OCC2', 'X1LOCALE']

# specific_metrics = combined.loc[combined.index.isin(specific_cols),
#                                  ['MI_score', 'RF_importance', 'Corr_with_target', 'MI_rank', 'RF_rank', 'Corr_rank', 'avg_rank']]

# print("\n=== METRICS FOR SPECIFIC COLUMNS ===")
# print(specific_metrics.sort_values('avg_rank').to_string())

Shape after dropping missing target: (17332, 2048)
Target value counts:
X4EVERDROP
0.0    14618
1.0     2714
Name: count, dtype: int64
Features after dropping high-missing cols: 1803

Computing Mutual Information...
Training Random Forest...
Computing Correlations...

=== TOP 100 FEATURES ===
               MI_score  RF_importance  Corr_with_target    avg_rank
X2DROPSTAT     0.173426       0.131708          0.501038    1.000000
X2ENROLSTAT    0.082775       0.041488          0.371879    2.666667
S2DROPOUTHS    0.057231       0.051086          0.389579    3.000000
X2UNIV2B       0.074932       0.030757          0.349872    4.000000
X2SQSTAT       0.054333       0.024262          0.288648    7.000000
X2ENRSTATSCH   0.069543       0.020866          0.261793    7.666667
S2ENROLLHS12   0.049736       0.025299          0.322491   10.333333
X2TXMTH        0.051326       0.007153          0.257827   14.666667
P1DROPOUT      0.037035       0.038422          0.324825   15.000000
X2TXMTH5       0

**Remove columns in which are imputed/not needed from the top 100 ranked columns**

After inspection, we found :
- imputed/not needed columns (37)
- date column in which have to be converted to month (1)
- a column in which we should remove and replace with the values it counts on (remove 1, add 6)

In [34]:
cols_to_keep = [
    "X2ENROLSTAT",
    "S2DROPOUTHS",
    "X2ENRSTATSCH",
    "X2SQSTAT",
    "S2ENROLLHS12",
    "X2TXMSCR",
    "X2TXMTH",
    "X2TXMTSCOR",
    "X2TXMPROF2",
    "X2TXMSEM",
    "P1DROPOUT",
    "X2REQLEVEL",
    "X2TXMPROF3",
    "S2SUREDIPL",
    "X2TXMPROF4",
    "X1TXMSCR",
    "X1TXMPROF3",
    "X1TXMPROF2",
    "X1TXMTSCOR",
    "X2TESTDATE",
    "X1TXMTH",
    "S1S8GRADE",
    "S1M8GRADE",
    "X1TXMPROF1",
    "C2AVGACTENG",
    "A2HIGHERED",
    "S2INSCHSUSP",
    "S2ALG1GRADE",
    "C2AVGSATMATH",
    "C2AVGACTSCI",
    "C2AVGACTCOMP",
    "X2TXMPROF1",
    "C2PERSISTYR1",
    "X2X1TXMSCR",
    "C2AVGSATREAD",
    "C2AVGACTREAD",
    "X2TXMPROF5",
    "S2REQTYP4YR",
    "C2AIDFLYER",
    "S2EDUEXP",
    "X2SES",
    "C2PCTEXAMINFO",
    "X2SQDATE",
    "C2AVGSATWRIT",
    "C2AVGACTMATH",
    "X2SES_U",
    "C2GRADPLAN",
    "C2INFOSESSN",
    "C2SELECTCLG",
    "S1SUREHSGRAD",
    "C2PCTINFO",
    "X2STUEDEXPCT",
    "C2UPMCLGREQ",
    "C2INFSTEM",
    "X2BEHAVEIN",
    "X1TXMPROF4",
    "C2UPMEOGEXAM",
    "C2UPMGRADREQ",
    "C2PCTEXAMPREP",
    "C2CLGAPP",
    "X1FAMINCOME",
    "X1PAR1EDU",
    "X1PAR2OCC2",
    "S2GRID1112",
    "X1PAR1OCC2",
    "X1PAR2EDU",
    "X1LOCALE",
    "X4EVERDROP"
]

print(len(cols_to_keep))

df_to_keep = df[[col for col in cols_to_keep if col in df.columns]]

df_to_keep.to_csv('cols_to_keep.csv', index = False)

68
